In [6]:
import os
import re
import math
import html
import webbrowser
import pandas as pd
import folium

from tqdm import tqdm
from folium.plugins import MarkerCluster


# 1. FILE CONFIGURATION


INPUT_FILE = (
    r"C:\Users\MARE\Documents\data311_project2\data\Latitude and Longitude Transit.csv"
)

OUTPUT_DIR = (
    r"C:\Users\MARE\OneDrive\Documents\data311_project2"
    r"\data\processed"
)

MAP_FILE = "transit_intersections_distance_map.html"
SUMMARY_FILE = "transit_distance_summary.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_MAP = os.path.join(OUTPUT_DIR, MAP_FILE)
SUMMARY_PATH = os.path.join(OUTPUT_DIR, SUMMARY_FILE)


# 2. MAP & ANCHOR CONFIGURATION


MAP_CENTER = [40.8196, -124.0617]
ZOOM_START = 11

# Destination Coordinates (Cal Poly Humboldt Library)
LIBRARY_LAT = 40.8196
LIBRARY_LON = -124.0617


# 3. HELPER FUNCTIONS


def calculate_haversine_distance(lat1, lon1, lat2, lon2):
    """Calculates the distance between two points in miles using Haversine formula."""
    if pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2):
        return None
    
    # Radius of earth in miles
    R = 3958.8 
    
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)
    
    a = (math.sin(delta_phi / 2) ** 2 + 
         math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda / 2) ** 2)
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    
    return round(R * c, 2)

def clean_text(series):
    return (
        series.astype("string")
        .str.strip()
        .replace("", pd.NA)
    )


# 4. LOAD & CLEAN DATA


print("\n" + "=" * 65)
print("LOADING & CLEANING TRANSIT DATA")
print("=" * 65)

df = pd.read_csv(
    INPUT_FILE,
    dtype={
        "location_id": "Int64",
        "geocode_location": "string",
        "geocode_location_type": "string",
        "ObjectId": "Int64",
        "longitude": "float64",
        "latitude": "float64"
    }
)

print(f"Loaded records: {len(df):,}")

# Clean location fields using text function from template
for col in ["geocode_location", "geocode_location_type"]:
    df[col] = clean_text(df[col])

# Drop rows missing coordinates
df = df.dropna(subset=["latitude", "longitude"]).copy()
print(f"Records with valid coordinate formatting: {len(df):,}")


# 5. DISTANCE MATHEMATICS & AGGREGATION


print("\n" + "=" * 65)
print("AGGREGATION & DISTANCE RESULTS")
print("=" * 65)

# Calculate distances directly using our math engine
df["distance_miles"] = df.apply(
    lambda row: calculate_haversine_distance(row["latitude"], row["longitude"], LIBRARY_LAT, LIBRARY_LON),
    axis=1
)

# Structure and group metrics using standard table aggregates
mapped = df.copy()

print(f"Original records: {len(mapped):,}")
print(f"Calculated markers to plot: {len(mapped):,}")
print(f"Unique transit stop locations: {mapped['geocode_location'].nunique():,}")

# Save distance calculations spreadsheet summary
mapped.to_csv(SUMMARY_PATH, index=False, encoding="utf-8")
print(f"Calculated path metrics saved to: {SUMMARY_PATH}")


# 6. GENERATE INTERACTIVE MAP STYLING (ESRI BASES ONLY)


print("\n" + "=" * 65)
print("GENERATING MAP ARCHITECTURE")
print("=" * 65)

# Initialize using the built-in, stable Esri World Street Map layer engine
m = folium.Map(
    location=MAP_CENTER, 
    zoom_start=ZOOM_START, 
    tiles="Esri.WorldStreetMap"
)

# Also layer Esri Imagery / Satellite views natively into the map framework
folium.TileLayer(
    tiles="Esri.WorldImagery",
    name="Esri World Imagery (Satellite)",
    control=True
).add_to(m)

# Plot Cal Poly Humboldt Library Destination Point
folium.Marker(
    location=[LIBRARY_LAT, LIBRARY_LON],
    popup=folium.Popup("<b>Cal Poly Humboldt Library</b><br>Destination Target Point", max_width=300),
    icon=folium.Icon(color="green", icon="book", prefix="fa"),
    tooltip="Destination: CPH Library"
).add_to(m)

# Target a 0.5 mile walking radius circle around campus library
folium.Circle(
    location=[LIBRARY_LAT, LIBRARY_LON],
    radius=804.67,  # 0.5 miles in meters
    color="#2E8B57",
    fill=True,
    fill_color="#2E8B57",
    fill_opacity=0.15,
    tooltip="0.5 Mile Buffer Zone"
).add_to(m)

# Cluster layer initialization
marker_cluster = MarkerCluster(name="Transit Intersections").add_to(m)

# Map features iterator loop
for _, row in tqdm(mapped.iterrows(), total=len(mapped), desc="Plotting Pins"):
    loc_id = row["location_id"]
    address = html.escape(str(row["geocode_location"]))
    loc_type = html.escape(str(row["geocode_location_type"]))
    lat = row["latitude"]
    lon = row["longitude"]
    dist = row["distance_miles"]
    
    popup_text = f"""
    <div style='font-family: Arial, sans-serif; font-size: 12px; width: 220px;'>
        <h4 style='margin: 0 0 5px 0; color: #333;'>Transit Point #{loc_id}</h4>
        <b>Type:</b> {loc_type}<br>
        <b>Location:</b> {address}<br>
        <hr style='margin: 8px 0; border: 0; border-top: 1px solid #ccc;'>
        <b style='color: #D62728;'>Distance to Library:</b> {dist} miles
    </div>
    """
    
    # Add coordinate point pins into cluster grouping
    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_text, max_width=250),
        icon=folium.Icon(color="blue", icon="bus", prefix="fa"),
        tooltip=f"ID {loc_id}: {dist} mi to Library"
    ).add_to(marker_cluster)
    
    # Draw vector dash lines tracking straight back to destination point
    folium.PolyLine(
        locations=[[lat, lon], [LIBRARY_LAT, LIBRARY_LON]],
        color="#2878B5",
        weight=1.5,
        opacity=0.4,
        dash_array="5, 5"
    ).add_to(m)

# Save finished map deployment layout
folium.LayerControl(collapsed=False).add_to(m)
m.save(OUTPUT_MAP)

print(f"\nInteractive Map successfully built and saved to:\n--> {OUTPUT_MAP}")
print("=" * 65 + "\n")



LOADING & CLEANING TRANSIT DATA
Loaded records: 100
Records with valid coordinate formatting: 100

AGGREGATION & DISTANCE RESULTS
Original records: 100
Calculated markers to plot: 100
Unique transit stop locations: 100
Calculated path metrics saved to: C:\Users\MARE\OneDrive\Documents\data311_project2\data\processed\transit_distance_summary.csv

GENERATING MAP ARCHITECTURE


Plotting Pins:   0%|          | 0/100 [00:00<?, ?it/s]

Plotting Pins: 100%|██████████| 100/100 [00:00<00:00, 5514.69it/s]


Interactive Map successfully built and saved to:
--> C:\Users\MARE\OneDrive\Documents\data311_project2\data\processed\transit_intersections_distance_map.html

